### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias

# Importaciones

In [51]:
using CSV, DataFrames, Glob, Statistics, Random, MLJ, PrettyTables, MLJModelInterface

# Preparación de los datos (20%)

## 1. Carga y unificación de los datos

In [33]:
base = "Datos Práctica"

# CSV del Investigador A
csv_inv_a = glob("Investigador A/day */*.csv", base)

# CSV del Investigador B
csv_inv_b = glob("Investigador B/*.csv", base)

all_csv = vcat(csv_inv_a, csv_inv_b)

dfs = [CSV.read(file, DataFrame) for file in all_csv]
df_total = vcat(dfs...)

println("Dataset correctamente cargado y unificado.")
println("Número de variables:         ", ncol(df_total))
println("Número de instancias:        ", nrow(df_total))
println("Número de individuos:        ", length(unique(df_total.subject)))
println("Número de clases de salida:  ", length(unique(df_total.Activity)))

Dataset correctamente cargado y unificado.
Número de variables:         563
Número de instancias:        10299
Número de individuos:        30
Número de clases de salida:  6


## 2. Análisis de valores ausentes

### Para analizar la calidad del dataset, se evaluó la presencia de valores ausentes tanto a nivel global como por variable. El porcentaje total de celdas con valores nulos en el dataset (`df_total`) es cercano al **1%**, lo cual indica un nivel de completitud muy alto. A nivel de columnas, **175 de las 563 variables** presentan al menos un valor faltante Las variables con mayor proporción de valores ausentes alcanzan ligeramente más del **10%**; es decir, de cada 100 observaciones en esas variables, alrededor de 10 están vacías. La gran mayoría de las variables no tienen datos faltantes, por lo que el problema de los nulos está concentrado en un subconjunto pequeño del total de features.

In [44]:
function nulls_analysis(df::DataFrame; top=10)

    # Porcentaje total de nulos
    total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
    total_values  = nrow(df) * ncol(df)
    pct_total = (total_missing / total_values) * 100
    println("Porcentaje de valores ausentes en el dataset: $(pct_total) %")

    # Porcentaje de nulos por columna
    df_nulls = DataFrame(
        Variable = names(df),
        NullsPercentage = [mean(ismissing.(df[!, c])) * 100 for c in names(df)]
    )

    # Variables con algún valor nulo
    df_nulls_pos = filter(:NullsPercentage => p -> p > 0, df_nulls)
    println("Variables con algún valor ausente: ", nrow(df_nulls_pos))

    # Ordenar de mayor a menor
    sort!(df_nulls_pos, :NullsPercentage, rev=true)

    # Mostrar top variables con más nulos
    if nrow(df_nulls_pos) > 0
        n_show = min(top, nrow(df_nulls_pos))
        println("\nTop $n_show variables con valores ausentes:")
        pretty_table(first(df_nulls_pos, n_show))
    else
        println("Ninguna columna contiene valores ausentes.")
    end

    return df_nulls_pos, pct_total
end

nulls_analysis(df_total);

Porcentaje de valores ausentes en el dataset: 0.9984242033534787 %
Variables con algún valor ausente: 175

Top 10 variables con valores ausentes:
┌──────────────────────────┬─────────────────┐
│                 Variable │ NullsPercentage │
│                   String │         Float64 │
├──────────────────────────┼─────────────────┤
│       tBodyGyroMag-mad() │         10.0301 │
│       tBodyGyroMag-iqr() │         10.0301 │
│         fBodyAcc-mad()-Y │         10.0204 │
│    fBodyAccJerk-mean()-X │         10.0204 │
│  tBodyAccJerk-energy()-X │          10.001 │
│ tBodyAccJerk-entropy()-Y │          10.001 │
│        tBodyAccMag-max() │          10.001 │
│     tGravityAccMag-std() │          10.001 │
│ tGravityAccMag-entropy() │          10.001 │
│         fBodyAcc-std()-X │          10.001 │
└──────────────────────────┴─────────────────┘


## 3. Tratamiento y transformación de datos

### En esta sección preparamos el conjunto de datos para su uso en los algoritmos de clasificación. El objetivo es obtener un dataset completamente limpio, sin valores ausentes y con la variable objetivo correctamente codificada. Queremos mantener una versión sin modificar del dataset, por lo que creamos una copia independiente llamada df_imputed. Como los datos pertenecen a diferentes individuos, la imputación se realiza por sujeto. Se emplean dos estrategias:

### a) Imputación para variables numéricas: mediana por individuo:
- ### Para cada sujeto, los valores numéricos ausentes se reemplazan por la mediana de esa misma variable dentro del sujeto.

### b) Imputación para variables no numéricas: moda por individuo:
- ### Las variables categóricas se imputan mediante el valor más frecuente dentro de cada sujeto.

In [46]:
df_imputed = deepcopy(df_total)

function impute_numeric_feature(df::DataFrame)
    individuals = groupby(df, :subject)

    for individual in individuals
        for col in names(individual)

            if col in (:subject, :Activity)
                continue
            end
        
            col_data = individual[!, col]

            if eltype(skipmissing(col_data)) <: Number #  Solo imputamos variables numéricas
                med = median(skipmissing(col_data))
                replace!(col_data, missing => med)
            end
        end
    end

    return df
end

function impute_non_numeric_feature(df::DataFrame)
    individuals = groupby(df, :subject)

    for individual in individuals
        for col in names(individual)

            if col in (:subject, :Activity)
                continue
            end
        
            col_data = individual[!, col]

            if !(eltype(skipmissing(col_data)) <: Number)
                mode_val = mode(skipmissing(col_data))
                replace!(col_data, missing => mode_val)
            end
        end
    end

    return df
end

impute_numeric_feature(df_imputed)
impute_non_numeric_feature(df_imputed)
nulls_analysis(df_imputed);

# La etiqueta debe ser categórica
df_imputed.Activity = categorical(df_imputed.Activity)

# Separamos features y target (características y etiqueta)
y = df_imputed.Activity
x = select(df_imputed, Not([:subject, :Activity]))

# Mostramos que estos tratamientos se aplican correctamente
println("\nTipo de la variable objetivo (y): $(eltype(df_imputed.Activity))")
println("Número de features (columnas en dataset de features): ", ncol(x))

Porcentaje de valores ausentes en el dataset: 0.0 %
Variables con algún valor ausente: 0
Ninguna columna contiene valores ausentes.

Tipo de la variable objetivo (y): CategoricalArrays.CategoricalValue{String31, UInt32}
Número de features (columnas en dataset de features): 561


## 4. Partición Holdout

### Para garantizar una evaluación estrictamente independiente, se realiza una partición hold-out basada en sujetos completos. El 10% de los individuos se reserva como test, y el 90% restante se usa para entrenamiento + validación. Esta división evita que datos del mismo sujeto aparezcan en distintos conjuntos, eliminando cualquier riesgo de data leakage.

In [48]:
subjects = unique(df_imputed.subject) # Sujetos únicos
Random.seed!(104)
shuffle!(subjects)
n_test = round(Int, length(subjects) * 0.10) # 10% de sujetos para test

test_subjects = subjects[1:n_test]
trainval_subjects = subjects[n_test+1:end]

df_trainval = filter(row -> row.subject in trainval_subjects, df_imputed); 
df_test = filter(row -> row.subject in test_subjects, df_imputed);

println("Sujetos en TEST (df_test): ", unique(df_test.subject))
println("Sujetos en TRAIN+VAL (df_trainval): ", unique(df_trainval.subject))

Sujetos en TEST (df_test): [25, 18, 22]
Sujetos en TRAIN+VAL (df_trainval): [1, 5, 7, 11, 3, 9, 23, 15, 17, 21, 13, 19, 27, 29, 2, 4, 6, 8, 10, 12, 14, 16, 20, 24, 26, 28, 30]


## 5. Cross-Validation individual wise

In [ ]:
function subject_folds(df::DataFrame; k::Int=5, seed::Int=104)
    subjects = unique(df.subject)
    Random.seed!(seed)
    shuffle!(subjects)

    # Asignamos un fold a cada sujeto, de forma más o menos equilibrada
    fold_id = Dict{eltype(subjects), Int}()
    for (i, s) in enumerate(subjects)
        fold_id[s] = 1 + (i - 1) % k
    end

    folds = Vector{Tuple{Vector{Int}, Vector{Int}}}(undef, k)

    for fold in 1:k # Construimos los folds
        train_idx = Int[]
        val_idx   = Int[]
        for (i, row) in enumerate(eachrow(df))
            if fold_id[row.subject] == fold
                push!(val_idx, i)
            else
                push!(train_idx, i)
            end
        end
        folds[fold] = (train_idx, val_idx)
    end

    return folds
end

# 6. Normalización Min-Max

In [ ]:
const MMI = MLJModelInterface

# Definición del modelo (wrapper)

struct MyMinMaxScaler <: MMI.Unsupervised
end

# Fase de entrenamiento (aprende min y max)
function MMI.fit(model::MyMinMaxScaler, verbosity::Int, X)
    
    # Convertimos X a matriz
    Xmat = MMI.matrix(X)

    # Por columna: min y max
    mins = mapslices(minimum, Xmat; dims=1)[1, :]
    maxs = mapslices(maximum, Xmat; dims=1)[1, :]

    # Guardamos en cache
    cache = (mins = mins, maxs = maxs)
    report = nothing

    return cache, report
end

# Transformación: aplicar (X - min) / (max - min)

function MMI.transform(model::MyMinMaxScaler, cache, X)
    Xmat = MMI.matrix(X)
    mins = cache.mins
    maxs = cache.maxs

    # Evitar división por cero
    ranges = maxs .- mins
    ranges[ranges .== 0] .= 1  # columnas constantes

    Xscaled = (Xmat .- mins) ./ ranges

    return Xscaled
end

# Modelos básicos y selección de atributos (20%)

- # Filtrado ANOVA

- # Filtrado Pearson